<a href="https://colab.research.google.com/github/jrhumberto/pep/blob/main/etl_eleitoral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook para extração de bases - Dimensão Eleitoral / IBGE

>Trata-se de notebook para extração de dados porovenientes das eleições do TSE dos anos de 2018 e 2022.

**Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

## Etapa Inicial: Instalação de Pacotes

In [43]:
install.packages("electionsBR")
install.packages("gtsummary")
install.packages("DT")
install.packages("gt")
install.packages("webshot2") # Install webshot2 package
library(electionsBR)
library(gtsummary)
library(DT)
library(gt)
library(webshot2) # Load webshot2 package

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘AsioHeaders’, ‘websocket’, ‘chromote’




### Fase 1 - Busca de resultados de Eleições de 2018 e 2022

In [30]:
library(dplyr)
library(readr)
library(electionsBR)

anos <- c(2018, 2022)
cols_interesse <- c("ANO_ELEICAO", "NR_TURNO", "SG_UF", "SG_UE","NM_UE","CD_CARGO","DS_CARGO","NR_CANDIDATO",
                    "NM_CANDIDATO","NM_URNA_CANDIDATO", "TP_AGREMIACAO","NR_PARTIDO","SG_PARTIDO","NM_PARTIDO",
                    "NM_FEDERACAO","SG_FEDERACAO","DS_COMPOSICAO_FEDERACAO", "NM_COLIGACAO",
                    "DS_COMPOSICAO_COLIGACAO", "DS_SIT_TOT_TURNO")

# Create the 'data' directory if it doesn't exist
dir.create("data", showWarnings = FALSE)

# Define the function to create electoral tables
criar_tabelas_eleicoes <- function(ano, cols_interesse, output_dir = "data") {
  # Baixa os dados
  dfe <- elections_tse(year = ano, type = "candidate")

  # Filtra e seleciona colunas
  tabela_eleicoes <- dfe %>%
    filter(DS_CARGO == "GOVERNADOR") %>%
    filter(DS_SIT_TOT_TURNO == "ELEITO") %>%
    select(any_of(cols_interesse)) %>%
    # Replace '#NULO#' with NA in character columns
    mutate(across(where(is.character), ~na_if(., "#NULO#")))

  # Salva como CSV com o ano no nome
  nome_arquivo <- paste0(output_dir, "/", "tabela_eleicoes_", ano, ".csv")
  write_csv(tabela_eleicoes, nome_arquivo)

  message("Arquivo salvo: ", nome_arquivo)
}

# Call the function for each year
for (ano in anos) {
  criar_tabelas_eleicoes(ano, cols_interesse, "data")
}

Processing the data...

Done.


Arquivo salvo: data/tabela_eleicoes_2018.csv

Processing the data...

Done.


Arquivo salvo: data/tabela_eleicoes_2022.csv



### Dicionário de Dados das Tabelas Eleitorais

| Campo                      | Descrição                                                                 |
| :------------------------- | :------------------------------------------------------------------------ |
| `ANO_ELEICAO`              | Ano da eleição.                                                           |
| `NR_TURNO`                 | Número do turno (1 para primeiro turno, 2 para segundo turno).             |
| `SG_UF`                    | Sigla da Unidade Federativa (Estado).                                     |
| `SG_UE`                    | Sigla da Unidade Eleitoral (Geralmente a mesma da UF para Governador).    |
| `NM_UE`                    | Nome da Unidade Eleitoral.                                                |
| `CD_CARGO`                 | Código do cargo disputado.                                                |
| `DS_CARGO`                 | Descrição do cargo (e.g., "GOVERNADOR").                                 |
| `NR_CANDIDATO`             | Número do candidato.                                                      |
| `NM_CANDIDATO`             | Nome completo do candidato.                                               |
| `NM_URNA_CANDIDATO`        | Nome do candidato que aparece na urna eletrônica.                         |
| `TP_AGREMIACAO`            | Tipo de agremiação (Partido ou Federação).                                |
| `NR_PARTIDO`               | Número do partido.                                                        |
| `SG_PARTIDO`               | Sigla do partido.                                                         |
| `NM_PARTIDO`               | Nome do partido.                                                          |
| `NM_FEDERACAO`             | Nome da federação, se aplicável.                                          |
| `SG_FEDERACAO`             | Sigla da federação, se aplicável.                                         |
| `DS_COMPOSICAO_FEDERACAO`  | Descrição da composição da federação.                                     |
| `NM_COLIGACAO`             | Nome da coligação, se aplicável.                                          |
| `DS_COMPOSICAO_COLIGACAO`  | Descrição da composição da coligação.                                     |
| `DS_SIT_TOT_TURNO`         | Situação do candidato no total de turnos (e.g., "ELEITO", "NÃO ELEITO"). |


### Fase 2 - Triagem dos campos úteis
- SG_UF
- ANO_ELEICAO
- DS_CARGO
- NM_CANDIDATO
- SG_PARTIDO


In [31]:
library(dplyr)
library(readr)

# Define a função para combinar os arquivos CSV
combinar_e_ordenar_eleicoes <- function(output_dir = "data/", output_filename = "eleicoes_2018_2022.csv") {

  # Colunas desejadas na ordem especificada
  cols_finais <- c("SG_UF", "ANO_ELEICAO", "DS_CARGO", "NM_CANDIDATO", "SG_PARTIDO")

  # Lista como Buffer para armazenar os dataframes de 2018 e 2022
  lista_df <- list()
  anos <- c(2018, 2022)

  for (ano in anos) {
    file_path <- paste0(output_dir, "tabela_eleicoes_", ano, ".csv")
    if (file.exists(file_path)) {
      df_ano <- read_csv(file_path)
      lista_df[[as.character(ano)]] <- df_ano
    } else {
      message(paste("Arquivo não encontrado para o ano", ano, ":", file_path))
    }
  }

  # Combina todos os dataframes da lista
  if (length(lista_df) > 0) {
    df_combinado <- bind_rows(lista_df)

    # Seleciona e reordena as colunas e ordena por SG_UF
    df_final <- df_combinado %>%
      select(all_of(cols_finais)) %>%
      arrange(SG_UF)

    # Salva o dataframe combinado e ordenado
    final_output_path <- paste0(output_dir, output_filename)
    write_csv(df_final, final_output_path)

    message("Arquivo combinado salvo: ", final_output_path)
    return(df_final)
  } else {
    message("Nenhum arquivo CSV encontrado para combinar.")
    return(NULL)
  }
}

# Chama a função para combinar os arquivos
eleicoes_2018_2022 <- combinar_e_ordenar_eleicoes()


Rows: 27 Columns: 20
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): SG_UF, SG_UE, NM_UE, DS_CARGO, NM_CANDIDATO, NM_URNA_CANDIDATO, TP...
dbl  (5): ANO_ELEICAO, NR_TURNO, CD_CARGO, NR_CANDIDATO, NR_PARTIDO
lgl  (3): NM_FEDERACAO, SG_FEDERACAO, DS_COMPOSICAO_FEDERACAO

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 27 Columns: 20
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (15): SG_UF, SG_UE, NM_UE, DS_CARGO, NM_CANDIDATO, NM_URNA_CANDIDATO, TP...
dbl  (5): ANO_ELEICAO, NR_TURNO, CD_CARGO, NR_CANDIDATO, NR_PARTIDO

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Arquivo combinado salvo: data/eleicoes_2018_2022.csv



### Fase 3 - Criação da variável austeridade, será 1 se o governador eleito for dos seguintes partidos:
- NOVO
- PL
- PSDB
- DEM
- UNIÂO
- PSD
- REPUBLICANOS

In [83]:
# Defina as siglas dos partidos que representam 'austeridade'
siglas_austeridade_fiscal <- c("NOVO", "PL", "PSDB", "REPUBLICANOS",
                               "DEM", "UNIÃO", "PSD")

# Crie a nova coluna 'austeridade'
eleicoes_2018_2022_austeridade <- eleicoes_2018_2022 %>%
  mutate(austeridade = ifelse(SG_PARTIDO %in% siglas_austeridade_fiscal, 1, 0))

# Salve o resultado em um novo arquivo CSV
output_path_austeridade <- "data/eleicoes_2018_2022_austeridade.csv"
write_csv(eleicoes_2018_2022_austeridade, output_path_austeridade)

message(paste0("Arquivo com coluna 'austeridade' salvo em: ", output_path_austeridade))


Arquivo com coluna 'austeridade' salvo em: data/eleicoes_2018_2022_austeridade.csv



O dataframe `eleicoes_2018_2022_austeridade` foi criado com a nova coluna `austeridade` e salvo em `data/eleicoes_2018_2022_austeridade.csv`.



---



In [84]:
library(dplyr)
library(readr)

# Remova as colunas 'NM_CANDIDATO' e 'DS_CARGO'
eleicoes_uf_2018_2022_austeridade <- eleicoes_2018_2022_austeridade %>%
  select(-NM_CANDIDATO, -DS_CARGO)

# Defina o caminho para o novo arquivo CSV
output_path_uf <- "data/eleicoes_uf_2018_2022_austeridade.csv"

# Salve o novo dataframe no arquivo CSV
write_csv(eleicoes_uf_2018_2022_austeridade, output_path_uf)

message(paste0("Arquivo com colunas removidas salvo em: ", output_path_uf))


Arquivo com colunas removidas salvo em: data/eleicoes_uf_2018_2022_austeridade.csv

